In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv("data/added_product_2016_04_28.csv")

In [ ]:
preds = pd.read_csv("preds.csv")

In [ ]:
preds.head()

In [ ]:
data.head()

In [ ]:
rename_products = {
    "ind_ahor_fin_ult1": "saving_account",
    "ind_aval_fin_ult1": "guarantees",
    "ind_cco_fin_ult1": "current_accounts",
    "ind_cder_fin_ult1": "derivada_account",
    "ind_cno_fin_ult1": "payroll_account",
    "ind_ctju_fin_ult1": "junior_account",
    "ind_ctma_fin_ult1": "mas_particular_account",
    "ind_ctop_fin_ult1": "particular_account",
    "ind_ctpp_fin_ult1": "particular_plus_account",
    "ind_deco_fin_ult1": "short_term_deposits",
    "ind_deme_fin_ult1": "medium_term_deposits",
    "ind_dela_fin_ult1": "long_term_deposits",
    "ind_ecue_fin_ult1": "e_account",
    "ind_fond_fin_ult1": "funds",
    "ind_hip_fin_ult1": "mortgage",
    "ind_plan_fin_ult1": "pensions",
    "ind_pres_fin_ult1": "loans",
    "ind_reca_fin_ult1": "taxes",
    "ind_tjcr_fin_ult1": "credit_card",
    "ind_valo_fin_ult1": "securities",
    "ind_viv_fin_ult1": "home_account",
    "ind_nomina_ult1": "payroll",
    "ind_nom_pens_ult1": "pensions",
    "ind_recibo_ult1": "direct_debit",
}

In [ ]:
data = data.replace(rename_products)

In [ ]:
data.head()

In [ ]:
data = data.merge(preds, on="ncodpers", how="left")

In [ ]:
data

In [ ]:
result = data.groupby(['ncodpers', 'xgb_preds'])['added_product'].apply(list).reset_index()

In [ ]:
result["xgb_preds"] = result["xgb_preds"].str.split()

In [ ]:
result

In [ ]:
# referring to https://github.com/benhamner/Metrics/blob/master/Python/ml_metrics/average_precision.py

def apk(actual, predicted, k=7):
    """
    Computes the average precision at k.
    This function computes the average prescision at k between two lists of
    items.
    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The average precision at k over the input lists
    """
    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)

    if not actual:
        return 0.0

    return score / min(len(actual), k)

def mapk(actual, predicted, k=7):
    """
    Computes the mean average precision at k.
    This function computes the mean average prescision at k between two lists
    of lists of items.
    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The mean average precision at k over the input lists
    """
    return np.mean([apk(a,p,k) for a, p in zip(actual, predicted)])

In [ ]:
result['mapk_score'] = result.apply(lambda row: apk(row.added_product, row.xgb_preds, 7), axis = 1)

In [ ]:
result.head(20)

In [ ]:
result.shape

In [ ]:
result['mapk_score'].mean()

In [ ]:
result['mapk_score'].min()

In [ ]:
#Coloque o tema darkgrid do seaborn
sns.set_style("darkgrid")

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(result['mapk_score'], bins=10, kde=True, color="g")
plt.xlabel("MAP@7")
plt.ylabel("Frequência")
#plt.title("Media de MAP@7 das recomendações")
plt.show()

In [ ]:
result[result['mapk_score'] == 1].shape[0] / result.shape[0]

In [ ]:
result["mapk_score"].describe()

In [ ]:
#cut into 5 quntile mapk_score (Sem Acerto, (0% - 25%], (25% - 50%], (50% - 75%], (75% - 100%])
result['quantile'] = pd.cut(result['mapk_score'], bins=[-np.inf, 0, 0.25, 0.5, 0.75, 1], labels=['Sem Acerto', '(0% - 25%]', '(25% - 50%]', '(50% - 75%]', '(75% - 100%]'])

In [ ]:
result["quantile"].value_counts()

In [ ]:
#Separe de 20 em 20% para plotar um grafico de barras baseado na variavel mapk_score
plt.figure(figsize=(10, 5))
sns.countplot(x="quantile", data=result, color="g")
plt.xlabel("MAP@7")
plt.ylabel("Frequência")
#plt.title("Porcentagem MAP@7 das recomendações")
plt.show()

In [ ]:
result[result['mapk_score'] > 0].shape[0] / result.shape[0] * 100

In [ ]:
result[result['mapk_score'] == 0].shape[0] / result.shape[0] * 100

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import confusion_matrix

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

compras_reais = result['added_product']
recomendacoes = result['xgb_preds'].apply(lambda x: x[:7])

df = pd.DataFrame({'Recomendacoes': recomendacoes, 'Compras': compras_reais})

# Inicialize listas para armazenar as métricas calculadas
precision_list = []
accuracy_list = []
recall_list = []
f1_score_list = []
specificity_list = []

for i in range(1, 8):
    # Obtenha as listas de recomendações e compras reais
    compras_reais = df['Compras'].tolist()
    recomendacoes = df['Recomendacoes'].apply(lambda x: x[:i]).tolist()

    # Inicialize as listas para armazenar as predições binárias (0 ou 1)
    y_true = []
    y_pred = []

    # Crie as listas de predições binárias com base nas recomendações e compras reais
    for compras, recs in zip(compras_reais, recomendacoes):
        # Transforme as listas em conjuntos para facilitar as operações de união e interseção
        compras_set = set(compras)
        recs_set = set(recs)

        # Se o produto recomendado estiver nas compras reais, a predição é verdadeira (1),
        # caso contrário, a predição é falsa (0)
        for prod in recs_set:
            y_true.append(1 if prod in compras_set else 0)
            y_pred.append(1)

        for prod in compras_set:
            if prod not in recs_set:
                y_true.append(1)
                y_pred.append(0)

    # Calcule as métricas para este valor de i e adicione-os às listas de resultados
    precision_list.append(precision_score(y_true, y_pred))
    accuracy_list.append(accuracy_score(y_true, y_pred))
    recall_list.append(recall_score(y_true, y_pred))
    f1_score_list.append(f1_score(y_true, y_pred))

# Agora você pode usar as listas de métricas para análise posterior
print("Accuracy:", accuracy_list)
print("Precision:", precision_list)
print("Recall:", recall_list)
print("F1 Score:", f1_score_list)


In [ ]:
#enable all columns
pd.set_option('display.max_columns', None)

In [ ]:
result.head()

In [ ]:
for i in range(1, 8):

    compras_reais = result['added_product']
    recomendacoes = result['xgb_preds'].apply(lambda x: x[:i])

    df = pd.DataFrame({'Recomendacoes': recomendacoes, 'Compras': compras_reais})

    # Inicializando as variáveis para as métricas
    total_recomendacoes = 0
    acertos = 0
    total_compras = 0
    recomendacoes_corretas = 0
    falsos_positivos = 0

    for idx, row in df.iterrows():
        recomendacoes = set(row['Recomendacoes'])
        compras = set(row['Compras'])

        total_recomendacoes += len(recomendacoes)
        total_compras += len(compras)

        for recom in recomendacoes:
            if recom in compras:
                acertos += 1
                recomendacoes_corretas += 1
            else:
                falsos_positivos += 1

    # Calculando as métricas
    acuracia = acertos / total_recomendacoes
    recall = recomendacoes_corretas / total_compras
    precisao = recomendacoes_corretas / total_recomendacoes
    especificidade = (total_compras - recomendacoes_corretas) / (total_compras - total_recomendacoes)
    f1_score = 2 * (precisao * recall) / (precisao + recall)

    print("\nMétricas para o top", i)
    print("Acurácia:", acuracia)
    print("Recall:", recall)
    print("Precisão:", precisao)
    print("Especificidade:", especificidade)
    print("F1-Score:", f1_score)

In [ ]:
for i in range(1, 8):

    compras_reais = result['added_product']
    recomendacoes = result['xgb_preds'].apply(lambda x: x[:i])

    df = pd.DataFrame({'Recomendacoes': recomendacoes, 'Compras': compras_reais})

    # Inicializando as variáveis para as métricas
    total_recomendacoes = 0
    acertos = 0
    total_compras = 0
    recomendacoes_corretas = 0
    compras_nao_recomendadas = 0
    falsos_positivos = 0

    for idx, row in df.iterrows():
        recomendacoes = set(row['Recomendacoes'])
        compras = set(row['Compras'])

        total_recomendacoes += len(recomendacoes)
        total_compras += len(compras)

        for recom in recomendacoes:
            if recom in compras:
                acertos += 1
                recomendacoes_corretas += 1
            else:
                compras_nao_recomendadas += 1
                falsos_positivos += 1

    # Calculando as métricas
    acuracia = acertos / total_recomendacoes
    recall = recomendacoes_corretas / total_compras
    precisao = recomendacoes_corretas / (recomendacoes_corretas + falsos_positivos)
    especificidade = (total_compras - recomendacoes_corretas) / (total_compras - total_recomendacoes)
    f1_score = 2 * (precisao * recall) / (precisao + recall)

    print("\nMétricas para o top", i)
    print("Acurácia:", acuracia)
    print("Recall:", recall)
    print("Precisão:", precisao)
    print("Especificidade:", especificidade)
    print("F1-Score:", f1_score)


In [ ]:
compras_reais = result['added_product']
recomendacoes = result['xgb_preds'].apply(lambda x: x[:7])

df = pd.DataFrame({'Recomendacoes': recomendacoes, 'Compras': compras_reais})

In [ ]:
# Função para calcular a matriz de confusão
def calculate_confusion_matrix(df):
    classes = sorted(list(set([item for sublist in df['Recomendacoes'] for item in sublist])))
    n_classes = len(classes)

    confusion = np.zeros((n_classes, n_classes), dtype=int)

    for _, row in df.iterrows():
        recomendacoes = set(row['Recomendacoes'])
        compras = set(row['Compras'])

        for recom_idx, recom_class in enumerate(classes):
            for compra_idx, compra_class in enumerate(classes):
                if recom_class in recomendacoes and compra_class in compras:
                    confusion[recom_idx][compra_idx] += 1

    return confusion, classes

# Calculando a matriz de confusão e as classes
confusion_matrix, classes = calculate_confusion_matrix(df)

print("Matriz de Confusão:")
plt.figure(figsize=(14, 6))
sns.heatmap(confusion_matrix, annot=True, fmt="d", xticklabels=classes, yticklabels=classes)

plt.show()
#print("\nClasses:", classes)